In [1]:
from tqdm.auto import tqdm       
import torch
from torch.utils.data import DataLoader
from custom_datasets.pokemon_dataset import PokemonDataset
from core.image_utils import compute_FID
from training.train_transformer import prepare_vae
from models.LightningDiT import LightningDiT


DEVICE = torch.device("cuda")
IMG_SIZE = 128
N_SAMPLES = 2**12
print("N_SAMPLES:", N_SAMPLES)

/home/msst/Utils/miniconda3/envs/qenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


N_SAMPLES: 4096


In [2]:
dataset_path = "/home/msst/repo/drifting/data/CelebA"
dataset_type = "FACES"

real_images_dataset = PokemonDataset(
    root_dir=dataset_path, 
    img_size=IMG_SIZE, 
    dataset_type=dataset_type
)

dataloader = DataLoader(
    real_images_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=16,
    pin_memory=True,
    drop_last=True
)

def get_real_images(dataloader, n_samples):
    n_batches = n_samples // 64
    real_images = []

    dataloader = iter(dataloader)
    for _ in tqdm(range(n_batches)):
        batch = next(dataloader)
        real_images.append(batch)
    
    real_images = torch.cat(real_images, dim=0)
    return real_images

real_images = get_real_images(dataloader, N_SAMPLES)

Number of images: 202599


100%|██████████| 64/64 [00:00<00:00, 107.43it/s]


In [3]:
vae_kwargs = {
    "device_map": "cuda:0",
    "torch_dtype": torch.bfloat16,
}
vae, latent_shape = prepare_vae(
    vae_name="flux_vae",
    path_to_vae="./checkpoints/flux_vae",
    img_size=IMG_SIZE,
    vae_kwargs=vae_kwargs
)

vit = LightningDiT.from_pretrained(
    "/home/msst/repo/drifting/checkpoints/CelebA/transformer_img128_ps2_flux-vae/transformer_step6000"
).to(torch.bfloat16).to(DEVICE).eval()


batch_size = 128
n_batches = N_SAMPLES // batch_size

gen_images = []

for b_id in tqdm(range(n_batches)):
    noise = torch.randn(batch_size, *latent_shape, dtype=torch.bfloat16, device=DEVICE)
    with torch.no_grad():
        gen_latent = vit(noise)
        gen_images.append(
            vae.decode(gen_latent).sample.detach().to(torch.float32).cpu()
        )
gen_images = torch.cat(gen_images, dim=0)

Model loaded from /home/msst/repo/drifting/checkpoints/CelebA/transformer_img128_ps2_flux-vae/transformer_step6000
  - Config: /home/msst/repo/drifting/checkpoints/CelebA/transformer_img128_ps2_flux-vae/transformer_step6000/config.json
  - Weights: /home/msst/repo/drifting/checkpoints/CelebA/transformer_img128_ps2_flux-vae/transformer_step6000/pytorch_model.pth


100%|██████████| 32/32 [00:15<00:00,  2.08it/s]


In [4]:
compute_FID(real_images, gen_images)

100%|██████████| 32/32 [00:03<00:00,  9.71it/s]


tensor(98.3608, device='cuda:0')